# Part 2 · Train one Condition-LoRA (run it once for `sketch`, once for `reference`)

Trains a small LoRA on the **4-bit** FLUX transformer using the cache from Part 1. Nothing else is on the GPU.

- **Needs:** a 16 GB GPU is enough in principle (T4 / Kaggle T4 / L4) · Part 1's cache · ~24 GB download
- **Produces:** `lora/Condition_LoRA/<COND>.safetensors` in your store
- **Two Kaggle GPUs:** open this notebook **twice**, one with `COND='sketch'` on one session and `COND='reference'` on another, so they run in parallel.
- **Do a smoke test first:** `SMOKE=True` runs 20 steps into a throw-away folder. It tells you seconds per step and GPU memory before you commit hours. Use it to set `STEPS`.
- **Time limit:** `MAX_MIN` makes it save and stop cleanly before the session limit. **Run the training cell again to resume** where it stopped; progress is kept in your store.
- **If it crashes with out-of-memory:** set `RES` smaller by rebuilding the cache, or tell me the error. **If you see "non-finite loss" many times on a T4:** fp16 is overflowing; that GPU class has no bf16 and this is the known risk of this route.
- With a big (A100/H100) GPU you can add `--no_quant --no_ckpt` to the training command.

In [4]:
COND      = 'sketch'      # 'sketch' or 'reference'
SMOKE     = True          # True: 20-step test run. Set False for the real run.
STEPS     = 1000          # real run length (optimizer steps)
MAX_MIN   = 150           # stop and save after this many minutes (0 = no limit). Re-run the cell to resume.
SAVE_EVERY = 250

In [5]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [6]:
# --- bootstrap (identical in every part): find the pipeline files, set paths ---
import os, glob, shutil, sys
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    hits = glob.glob('/kaggle/input/**/common.py', recursive=True)
    assert hits, 'Upload the pipeline/*.py files + pilot_v1.zip as a Kaggle dataset and add it as an input to this notebook.'
    SRC, WORK = os.path.dirname(hits[0]), '/kaggle/working'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    SRC, WORK = '/content/drive/MyDrive/sketchy_dataset_one_sketch/pipeline', '/content'
os.makedirs(f'{WORK}/pipeline', exist_ok=True)
for f in glob.glob(f'{SRC}/*.py'):
    shutil.copy(f, f'{WORK}/pipeline/')
sys.path.insert(0, f'{WORK}/pipeline')
from common import *
print('platform:', 'Kaggle' if ON_KAGGLE else 'Colab', '| STORE =', STORE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
platform: Colab | STORE = /content/drive/MyDrive/sketchy_dataset_one_sketch


In [7]:
!pip install -q diffusers==0.33.1 transformers==4.52.4 peft==0.15.2 accelerate==1.7.0 bitsandbytes sentencepiece protobuf opencv-python-headless ipdb

In [8]:
import torch
p = torch.cuda.get_device_properties(0)
print(p.name, round(p.total_memory / 1e9), 'GB | bf16:', torch.cuda.get_device_capability()[0] >= 8)

Tesla T4 16 GB | bf16: False


In [9]:
import glob, shutil, os
DRIVE = '/content/drive/MyDrive/sketchy_dataset_one_sketch/pipeline'
print('Drive copy has patch:', open(f'{DRIVE}/apply_patches.py').read().count('_uc_run'))
!rm -rf /content/pipeline /content/UniCombine
os.makedirs('/content/pipeline', exist_ok=True)
for f in glob.glob(f'{DRIVE}/*.py'):
    shutil.copy(f, '/content/pipeline/')
print('Colab copy has patch:', open('/content/pipeline/apply_patches.py').read().count('_uc_run'))

Drive copy has patch: 4
Colab copy has patch: 4


In [10]:
import os
import glob
import shutil
import subprocess
import sys

# Ensure all pipeline/patch files from your Drive are copied to the local runtime pipeline directory
source_pipeline_dir = '/content/drive/MyDrive/sketchy_dataset_one_sketch/pipeline'
target_pipeline_dir = '/content/pipeline'

os.makedirs(target_pipeline_dir, exist_ok=True)
if os.path.exists(source_pipeline_dir):
    print(f"Copying missing files from {source_pipeline_dir}...")
    for f in glob.glob(f'{source_pipeline_dir}/*'):
        shutil.copy(f, target_pipeline_dir)
    print("Files copied successfully:", os.listdir(target_pipeline_dir))
else:
    print(f"Warning: Source folder {source_pipeline_dir} not found. Please verify the folder path in Google Drive.")

# Let's run the repository cloning first if it does not exist
UNI = '/content/UniCombine'
if not os.path.exists(UNI):
    print("Cloning UniCombine repository...")
    subprocess.check_call(["git", "clone", "-q", "https://github.com/Xuan-World/UniCombine.git", UNI])

# Run the patch script with BOTH required arguments (UNI and target_pipeline_dir)
print("Applying patches...")
result = subprocess.run(
    [sys.executable, f"{target_pipeline_dir}/apply_patches.py", UNI, target_pipeline_dir],
    capture_output=True,
    text=True
)

print("--- PATCH STDOUT ---")
print(result.stdout)
print("--- PATCH STDERR ---")
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"Patch script failed with exit code: {result.returncode}")
else:
    print("Patches applied successfully!")

Copying missing files from /content/drive/MyDrive/sketchy_dataset_one_sketch/pipeline...
Files copied successfully: ['P3_train_union_lora.ipynb', 'sketchy_loader.py', 'eval_metrics.py', 'cache_build.py', 'README.md', 'P5_evaluate.ipynb', '01_freeze_pilot_dataset.ipynb', 'P2_train_condition_lora.ipynb', '02_unicombine_pipeline.ipynb', 'common.py', 'apply_patches.py', 'train_lowmem.py', 'P4_generate.ipynb', 'P1_cache.ipynb', 'unicombine_infer.py']
Cloning UniCombine repository...
Applying patches...
--- PATCH STDOUT ---
patch 1 applied (reference position ids)
patch 2 applied (gradient checkpointing)
UniCombine is patched.

--- PATCH STDERR ---

Patches applied successfully!


In [11]:
!grep -c "_uc_run" /content/UniCombine/src/UniCombineTransformer2DModel.py
!grep -c "_uc_run" /content/pipeline/apply_patches.py

3
4


In [12]:
hf_login()   # FLUX.1-schnell is gated: accept the terms on its Hugging Face page first, then use a read token

In [13]:
download('black-forest-labs/FLUX.1-schnell', FLUX, allow=DENOISE_PARTS)                  # ~24 GB, the 4-bit conversion happens while loading
download('ostris/FLUX.1-schnell-training-adapter', ASSIST)
CACHE = need('cache_r256.pt')
print(CACHE)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

/content/drive/MyDrive/sketchy_dataset_one_sketch/cache/cache_r256.pt


In [14]:
OUT = f'{STORE}/runs_smoke' if SMOKE else f'{STORE}/runs'
n = 20 if SMOKE else STEPS
!python {PIPE}/train_lowmem.py --uni_dir {UNI} --flux_dir {FLUX} --assistant_dir {ASSIST} --cache {CACHE} \
    --cond_types {COND} --out_name {COND} --out_dir {OUT} --steps {n} --save_every {SAVE_EVERY if not SMOKE else 10} --max_minutes {MAX_MIN}

I0000 00:00:1792761703.543646    4991 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792761705.936151    4991 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
GPU Tesla T4 | compute dtype torch.float16 | 4-bit base: True | checkpointing: True
160 training samples, latent (16, 32, 32), text tokens 128
Loading checkpoint shards:   0% 0/3 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "/content/pipeline/train_lowmem.py", line 207, in <module>
    main()
    ~~~~^^
  File "/content/pipeline/train_lowmem.py", line 71, in main
    tr = UniCombineTransformer2DModel.from_pretrained(os.path.join(a.flux_dir, "transformer"), torch_dtype=dt, quantization_config=qc)
  File "/usr/local/lib/python3.13/dist-packages/huggingfac

In [15]:
src = f'{OUT}/exports/{COND}.safetensors'
assert os.path.exists(src), 'Not finished yet: run the training cell again to resume.'
if SMOKE:
    print('smoke test OK. Now set SMOKE=False, choose STEPS from the s/step above, and run again.')
else:
    os.makedirs(f'{STORE}/lora/Condition_LoRA', exist_ok=True)
    shutil.copy(src, f'{STORE}/lora/Condition_LoRA/{COND}.safetensors')
    print('saved', f'{STORE}/lora/Condition_LoRA/{COND}.safetensors')

AssertionError: Not finished yet: run the training cell again to resume.